In [2]:
import tensorflow as tf

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train.shape, x_test.shape

((60000, 28, 28), (10000, 28, 28))

In [3]:
# 정답 데이터 분포 확인
import numpy as np

np.unique(y_train, return_counts=True), np.unique(y_test, return_counts=True)

((array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=uint8),
  array([5923, 6742, 5958, 6131, 5842, 5421, 5918, 6265, 5851, 5949])),
 (array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=uint8),
  array([ 980, 1135, 1032, 1010,  982,  892,  958, 1028,  974, 1009])))

In [4]:
# 28*28 => 784 1차원, 정규화(/255.0)
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Flatten, Rescaling

model = Sequential([
    Input(shape=(28, 28)), # 입력
    Rescaling(1./255.),    # 정규화
    Flatten(),             # 1차원으로 변환
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax'),
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 28, 28)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# loss = "categorical_crossentropy" # one-hot 인코딩된 레이블 사용
# loss = "sparse_categorical_crossentropy" # 정수 레이블 사용

optimizer = keras.optimizers.Adam(learning_rate=0.001)
loss = keras.losses.SparseCategoricalCrossentropy() # 정수 레이블 사용
model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

In [15]:
# 모델 체크포인트 콜백 설정 (최고 성능 모델 저장)
model_checkpoint = keras.callbacks.ModelCheckpoint(
    './pkl/20260101_handwritten_best.keras', 
    monitor='val_accuracy',
    save_best_only=True
)


In [14]:
# 원래 스케일링을 해줘야 하나 여기서는 시간상 그냥 생략했다
history = model.fit(
    x_train, 
    y_train, 
    validation_data=(x_test, y_test), 
    epochs=50, 
    verbose=2,
    callbacks=[model_checkpoint]
)

Epoch 1/50
1875/1875 - 3s - 2ms/step - accuracy: 0.9985 - loss: 0.0052 - val_accuracy: 0.9771 - val_loss: 0.2034
Epoch 2/50
1875/1875 - 5s - 3ms/step - accuracy: 0.9983 - loss: 0.0064 - val_accuracy: 0.9794 - val_loss: 0.1922
Epoch 3/50
1875/1875 - 3s - 2ms/step - accuracy: 0.9983 - loss: 0.0062 - val_accuracy: 0.9783 - val_loss: 0.2149
Epoch 4/50
1875/1875 - 3s - 2ms/step - accuracy: 0.9982 - loss: 0.0067 - val_accuracy: 0.9795 - val_loss: 0.2014
Epoch 5/50
1875/1875 - 3s - 2ms/step - accuracy: 0.9988 - loss: 0.0038 - val_accuracy: 0.9780 - val_loss: 0.2140
Epoch 6/50
1875/1875 - 3s - 2ms/step - accuracy: 0.9988 - loss: 0.0042 - val_accuracy: 0.9805 - val_loss: 0.2355
Epoch 7/50
1875/1875 - 3s - 2ms/step - accuracy: 0.9979 - loss: 0.0087 - val_accuracy: 0.9804 - val_loss: 0.1852
Epoch 8/50
1875/1875 - 3s - 2ms/step - accuracy: 0.9992 - loss: 0.0026 - val_accuracy: 0.9786 - val_loss: 0.2050
Epoch 9/50
1875/1875 - 4s - 2ms/step - accuracy: 0.9980 - loss: 0.0068 - val_accuracy: 0.9776 - 

In [17]:
model.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9779 - loss: 0.3055


[0.3055485188961029, 0.9779000282287598]

In [8]:
y_pred = model.predict(x_test)
x_test.shape

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 977us/step


(10000, 28, 28)

In [9]:
# 예측데이터와 실제 데이터비교
import numpy as np
np.argmax(y_pred, axis=1), y_test

(array([7, 2, 1, ..., 4, 5, 6], shape=(10000,)),
 array([7, 2, 1, ..., 4, 5, 6], shape=(10000,), dtype=uint8))

In [16]:
model.save("./pkl/20260601_handwritten.keras")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, np.argmax(y_pred, axis=1))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()